# L12 · 海龟策略（唐奇安通道 + ATR 仓位）

**预计时长**：90 min | **难度**：⭐⭐⭐⭐ | **前置**：L11

## 本节目标
1. 唐奇安通道：N 日最高/最低突破系统
2. ATR（Average True Range）：真实波动幅度，仓位管理的基石
3. 原版海龟的仓位公式：`unit = risk_per_trade / (atr × point_value)`
4. 对比 qtrader 的 TurtleStrategy（含 atr_n 参数版）

## 第 1 段：金融概念

### 唐奇安通道（Donchian Channel）
- 上轨 = 过去 N 日最高价
- 下轨 = 过去 M 日最低价
- **突破上轨 → 买入；跌破下轨 → 卖出**
- N=20, M=10 是 1980 年代「海龟交易实验」原版参数

### ATR（Average True Range）
当日「真实波幅」TR 取以下三者的最大值：
1. 今日最高 - 今日最低
2. |今日最高 - 昨日收盘|
3. |今日最低 - 昨日收盘|

ATR(N) = TR 的 N 日均值，衡量「日均波动金额」。

### 原版海龟仓位公式（核心创新）
```
unit = (account × risk_per_trade) / (atr × dollar_per_point)
```
- risk_per_trade：单次交易风险，通常 1%
- 含义：价格波动 1 ATR 时，账户损失正好 = risk_per_trade
- 让「单次亏损可控」成为第一性原则

### 为什么 A 股版要调整
- 原版海龟做多也做空（期货思维）
- A 股现货只能做多 → 仓位 ∈ [0, max_pos]
- qtrader.TurtleStrategy 用 `atr_n=None` 时退化为 [0,1] 二值

In [ ]:
import sys
from pathlib import Path

# 自动定位 phase1_foundation（共享 _data/_style）+ project root
_cwd = Path.cwd()
_p1 = _cwd if (_cwd / '_data.py').exists() else (
    _cwd / 'learning' / 'phase1_foundation'
    if (_cwd / 'learning' / 'phase1_foundation' / '_data.py').exists()
    else _cwd.parent / 'phase1_foundation'
)
sys.path.insert(0, str(_p1))
_proj = _p1.parent.parent if _p1.name == 'phase1_foundation' else _p1
if (_proj / 'qtrader' / '__init__.py').exists():
    sys.path.insert(0, str(_proj))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import _style
from _data import get_stock_data
from qtrader import (
    BacktestEngine, CostModel,
    DualMAStrategy, TurtleStrategy, GridStrategy,
    print_comparison_table,
)

## 第 2 段：手写唐奇安通道信号

In [ ]:
byd = get_stock_data('002594').set_index('date')

def donchian_signal(df: pd.DataFrame, entry_n: int = 20, exit_n: int = 10) -> pd.Series:
    \"\"\"唐奇安通道突破信号（向量化）。\"\"\"
    upper = df['high'].rolling(entry_n).max().shift(1)  # 用昨日通道
    lower = df['low'].rolling(exit_n).min().shift(1)
    buy = df['close'] > upper
    sell = df['close'] < lower

    signal = pd.Series(np.nan, index=df.index)
    signal[buy] = 1.0
    signal[sell] = 0.0
    # fillna + shift(1)：避免未来函数
    return signal.ffill().fillna(0).shift(1).fillna(0)

sig = donchian_signal(byd, entry_n=20, exit_n=10)
print(f"持仓占比: {sig.mean():.2%}")
print(f"信号切换次数: {(sig.diff().abs() > 1e-6).sum()}")

## 第 3 段：ATR 计算与可视化

In [ ]:
def compute_atr(df: pd.DataFrame, n: int = 20) -> pd.Series:
    \"\"\"计算 ATR（N 日均真实波幅）。\"\"\"
    prev_close = df['close'].shift(1)
    tr = pd.concat([
        df['high'] - df['low'],
        (df['high'] - prev_close).abs(),
        (df['low'] - prev_close).abs(),
    ], axis=1).max(axis=1)
    return tr.rolling(n).mean()

atr = compute_atr(byd, n=20)
atr_pct = (atr / byd['close']) * 100  # 转换为百分比

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 7), sharex=True,
                               gridspec_kw={'height_ratios': [3, 1]})
byd['close'].plot(ax=ax1, label='收盘价', alpha=0.7)
byd['high'].rolling(20).max().shift(1).plot(ax=ax1, label='上轨 N=20', linestyle='--')
byd['low'].rolling(10).min().shift(1).plot(ax=ax1, label='下轨 M=10', linestyle='--')
ax1.set_title('比亚迪 + 唐奇安通道')
ax1.legend()

atr_pct.plot(ax=ax2, color='purple', label='ATR%')
ax2.set_ylabel('日波动率(%)')
ax2.legend()
plt.tight_layout(); plt.show()

print(f"ATR% 平均: {atr_pct.mean():.2f}%，最大: {atr_pct.max():.2f}%")

## 第 4 段：ATR 仓位管理 + 对比框架版

In [ ]:
# 手写 ATR 仓位
risk_per_trade = 0.01  # 1% 单次风险
max_pos = 1.0
sized = (risk_per_trade / (atr_pct / 100).replace(0, np.nan)).clip(0, max_pos).fillna(0)
position = sig * sized  # 仅持仓状态应用 ATR 仓位

# qtrader 框架版
engine = BacktestEngine()
result_binary = engine.run(byd.reset_index(), TurtleStrategy(entry_n=20, exit_n=10))
result_atr = engine.run(byd.reset_index(),
    TurtleStrategy(entry_n=20, exit_n=10, atr_n=20, risk_per_trade=0.01, max_pos=1.0))

fig, ax = plt.subplots(figsize=(13, 5))
(1 + byd['close'].pct_change().fillna(0)).cumprod().plot(ax=ax, label='买入持有', alpha=0.5)
result_binary.nav.plot(ax=ax, label='Turtle 二值', linestyle='--')
result_atr.nav.plot(ax=ax, label='Turtle + ATR', linewidth=1.5)
ax.set_title('海龟二值 vs ATR 仓位版')
ax.legend()
plt.tight_layout(); plt.show()

print(f"二值版夏普: {result_binary.metrics['sharpe']:.3f}")
print(f"ATR 版夏普:  {result_atr.metrics['sharpe']:.3f}")
print(f"ATR 版持仓时长: {result_atr.metrics['exposure_time']:.2%}")

## 第 5 段：海龟策略的人性化解读

- **纪律性**：突破就买，跌破就卖，不带感情——1980 年代实验证明规则化交易胜过直觉
- **仓位智慧**：ATR 高（波动大）→ 减仓；ATR 低（平静）→ 加仓
- **A 股实战注意**：
  - 涨停一字板无法买入，回测里的「突破买入」可能实盘做不到
  - 长期熊市（2010-2014、2021-2023）海龟会持续小亏，心理考验大
  - 必须搭配 ETF 或一篮子股票做分散，否则单股黑天鹅致命

### 🔮 下节 L13：网格策略
与趋势跟踪相反，网格是「震荡市利器」，在 A 股的高波动环境有独特价值。